# 🏭 Punto 1: Clasificación de Piezas Industriales
## Paso 1: Exploración de Datos

**Objetivo:** Explorar el dataset de piezas industriales de Kaggle para entender:
- Número de categorías/clases
- Distribución de imágenes por clase
- Dimensiones de las imágenes
- Calidad de los datos

### 📚 Importar Librerías

In [ ]:
# Librerías básicas
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

# Librerías para imágenes
from PIL import Image
import cv2

# Configuración
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Configurar tamaño de figuras
plt.rcParams['figure.figsize'] = (12, 6)

print('✅ Librerías importadas correctamente')

### 📂 Definir Rutas de Datos

In [ ]:
# Ruta base del proyecto
BASE_DIR = Path(r'C:\Users\Diego\Documents\Maestria\tallerML')
DATA_DIR = BASE_DIR / 'datos' / 'DataSet'

# Rutas de los datasets
TRAIN_DIR = DATA_DIR / 'Train_Dataset'
VALID_DIR = DATA_DIR / 'Valid_Dataset'
TEST_DIR = DATA_DIR / 'Test_Dataset'

# Verificar que existen
print(f'📁 Directorio de datos: {DATA_DIR}')
print(f'Existe TRAIN: {TRAIN_DIR.exists()}')
print(f'Existe VALID: {VALID_DIR.exists()}')
print(f'Existe TEST: {TEST_DIR.exists()}')

### 🔍 Explorar Estructura del Dataset

Vamos a entender cómo están organizados los datos

In [ ]:
# Listar contenido de Train_Dataset
print('📋 Contenido de Train_Dataset:')
for item in TRAIN_DIR.iterdir():
    print(f'  - {item.name}')

In [ ]:
# Leer el archivo labels.csv para entender las categorías
labels_path = TRAIN_DIR / 'labels.csv'

if labels_path.exists():
    # Leer solo las primeras filas para ver la estructura
    df_labels = pd.read_csv(labels_path, nrows=10)
    print('🏷️ Primeras filas del archivo labels.csv:')
    display(df_labels)
    
    print('\n📊 Columnas disponibles:')
    print(df_labels.columns.tolist())

In [ ]:
# Cargar el dataset completo de etiquetas
df_labels_full = pd.read_csv(TRAIN_DIR / 'labels.csv')

print(f'📊 Total de imágenes en entrenamiento: {len(df_labels_full):,}')
print(f'\nInformación del dataset:')
df_labels_full.info()

### 📈 Análisis de Categorías

Identificar cuántas clases tenemos y su distribución

In [ ]:
# Identificar la columna de categorías
category_col = [col for col in df_labels_full.columns if 'categ' in col.lower() or 'label' in col.lower() or 'class' in col.lower()]

if category_col:
    category_col = category_col[0]
    print(f'✅ Columna de categorías identificada: {category_col}')
else:
    print('⚠️ Revisemos todas las columnas:')
    print(df_labels_full.columns.tolist())
    category_col = df_labels_full.columns[-1]
    print(f'Usando: {category_col}')

In [ ]:
# Contar imágenes por categoría
category_counts = df_labels_full[category_col].value_counts()

print(f'🏭 Número de categorías: {len(category_counts)}')
print(f'\n📊 Distribución de imágenes por categoría:\n')
print(category_counts)

In [ ]:
# Visualizar distribución de clases
plt.figure(figsize=(14, 6))

# Gráfico de barras
ax = category_counts.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Distribución de Imágenes por Categoría (Training Set)', fontsize=16, fontweight='bold')
plt.xlabel('Categoría', fontsize=12)
plt.ylabel('Número de Imágenes', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)

# Añadir valores encima de las barras
for i, v in enumerate(category_counts):
    ax.text(i, v + 10, str(v), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Análisis de balance
print('\n⚖️ Análisis de Balance de Clases:')
print(f'Clase más frecuente: {category_counts.index[0]} ({category_counts.iloc[0]} imágenes)')
print(f'Clase menos frecuente: {category_counts.index[-1]} ({category_counts.iloc[-1]} imágenes)')
print(f'Ratio desbalance: {category_counts.iloc[0] / category_counts.iloc[-1]:.2f}x')

### 🖼️ Explorar Imágenes

Analizar dimensiones, formatos y visualizar ejemplos

In [ ]:
# Ruta de las imágenes
images_dir = TRAIN_DIR / 'images'

# Obtener lista de imágenes
image_files = list(images_dir.glob('*.jpg')) + list(images_dir.glob('*.png'))

print(f'🖼️ Total de archivos de imágenes encontrados: {len(image_files):,}')

In [ ]:
# Analizar dimensiones de una muestra de imágenes
sample_size = min(500, len(image_files))
sample_images = np.random.choice(image_files, size=sample_size, replace=False)

image_dimensions = []

print(f'📏 Analizando dimensiones de {sample_size} imágenes aleatorias...')

for img_path in sample_images:
    try:
        img = Image.open(img_path)
        image_dimensions.append(img.size)
    except Exception as e:
        print(f'⚠️ Error al leer {img_path.name}: {e}')

# Convertir a DataFrame
df_dims = pd.DataFrame(image_dimensions, columns=['width', 'height'])

print('\n📊 Estadísticas de dimensiones:')
print(df_dims.describe())

In [ ]:
# Visualizar distribución de dimensiones
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histograma de anchos
axes[0].hist(df_dims['width'], bins=30, color='skyblue', edgecolor='black')
axes[0].set_title('Distribución de Anchos', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Ancho (píxeles)')
axes[0].set_ylabel('Frecuencia')
axes[0].grid(alpha=0.3)

# Histograma de alturas
axes[1].hist(df_dims['height'], bins=30, color='lightcoral', edgecolor='black')
axes[1].set_title('Distribución de Alturas', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Alto (píxeles)')
axes[1].set_ylabel('Frecuencia')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Dimensiones más comunes
print('\n📐 Dimensiones más comunes:')
common_dims = df_dims.value_counts().head(10)
print(common_dims)

### 🎨 Visualizar Muestras de Cada Categoría

In [ ]:
# Función para visualizar muestras
def plot_sample_images(df, category_col, images_dir, n_categories=5, n_images=4):
    categories = df[category_col].unique()[:n_categories]
    
    fig, axes = plt.subplots(n_categories, n_images, figsize=(15, 3*n_categories))
    
    for i, category in enumerate(categories):
        category_images = df[df[category_col] == category]
        sample_images = category_images.sample(min(n_images, len(category_images)))
        
        for j, (idx, row) in enumerate(sample_images.iterrows()):
            img_filename = row[0] if 'filename' not in row else row['filename']
            img_path = images_dir / str(img_filename)
            
            if img_path.exists():
                img = Image.open(img_path)
                axes[i, j].imshow(img)
                axes[i, j].axis('off')
                
                if j == 0:
                    axes[i, j].set_title(f'{category}\n{img.size[0]}x{img.size[1]}', 
                                        fontsize=10, fontweight='bold')
                else:
                    axes[i, j].set_title(f'{img.size[0]}x{img.size[1]}', fontsize=9)
    
    plt.suptitle('Muestras de Imágenes por Categoría', fontsize=16, fontweight='bold', y=1.0)
    plt.tight_layout()
    plt.show()

# Llamar a la función
plot_sample_images(df_labels_full, category_col, images_dir, n_categories=5, n_images=4)

### 📋 Resumen de la Exploración

In [ ]:
print('='*60)
print('📊 RESUMEN DE EXPLORACIÓN DE DATOS')
print('='*60)
print(f'\n✅ Dataset de Entrenamiento:')
print(f'   - Total de imágenes: {len(df_labels_full):,}')
print(f'   - Número de categorías: {len(category_counts)}')
print(f'   - Dimensiones promedio: {int(df_dims["width"].mean())}x{int(df_dims["height"].mean())}')
print(f'\n⚖️ Balance de clases:')
print(f'   - Clase más frecuente: {category_counts.iloc[0]} imágenes')
print(f'   - Clase menos frecuente: {category_counts.iloc[-1]} imágenes')
print(f'   - Ratio: {category_counts.iloc[0] / category_counts.iloc[-1]:.2f}x')
print(f'\n🎯 Recomendaciones para el modelo:')
print(f'   1. Normalizar todas las imágenes a 224x224 (VGG16 standard)')
print(f'   2. Aplicar Data Augmentation para balancear clases')
print(f'   3. Usar Transfer Learning con VGG16 preentrenado')
print(f'   4. Considerar weights de clase para el desbalance')
print('='*60)

### 💾 Guardar Resultados

In [ ]:
import json

# Crear carpeta para resultados
results_dir = Path(r'C:\Users\Diego\Documents\Maestria\tallerML\proyecto_ml_aws\resultados')
results_dir.mkdir(exist_ok=True)

# Guardar estadísticas
stats = {
    'total_images': len(df_labels_full),
    'num_categories': len(category_counts),
    'category_distribution': category_counts.to_dict(),
    'avg_width': int(df_dims['width'].mean()),
    'avg_height': int(df_dims['height'].mean())
}

with open(results_dir / 'exploracion_datos.json', 'w') as f:
    json.dump(stats, f, indent=4)

print('✅ Resultados guardados en:', results_dir)

---
### 🎯 Siguiente Paso

**Notebook 02:** Desarrollo del Modelo CNN con Transfer Learning (VGG16)